# SmartBugs Wild Dataset Validation
## Day 2: Learning by Doing - ML Part

This notebook validates the SmartBugs Wild dataset to ensure it's suitable for training our vulnerability detection model.

### Goals:
1. Understand dataset structure and size
2. Check for vulnerabilities labels (CRITICAL!)
3. Validate data quality
4. Identify potential issues
5. Prepare data preprocessing strategy

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import json

sns.set_style('whitegrid')
%matplotlib inline

## 1. Dataset Overview

In [ ]:
DATA_DIR = Path('../data/smartbugs-wild')
CONTRACTS_DIR = DATA_DIR / 'contracts'
CSV_PATH = DATA_DIR / 'contracts.csv'

print(f"Dataset directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
print(f"Contracts directory: {CONTRACTS_DIR}")
print(f"Contracts directory exists: {CONTRACTS_DIR.exists()}")
print(f"CSV file exists: {CSV_PATH.exists()}")

In [ ]:
# Load metadata CSV
df = pd.read_csv(CSV_PATH)
print(f"\nDataset shape: {df.shape}")
print(f"Number of contracts: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(10)

In [ ]:
# Check actual contract files
contract_files = list(CONTRACTS_DIR.glob('*.sol'))
print(f"\nNumber of .sol files: {len(contract_files):,}")
print(f"Match with CSV records: {len(contract_files) == len(df)}")

if len(contract_files) > 0:
    print(f"\nSample contract files:")
    for f in contract_files[:5]:
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")

## 2. CRITICAL: Check for Vulnerability Labels

This is the most important check. We need labeled data for supervised learning!

In [ ]:
# Check if there are any vulnerability-related columns
print("Columns in dataset:")
for col in df.columns:
    print(f"  - {col}")

print("\n" + "="*80)
print("🚨 CRITICAL FINDING:")
print("="*80)

vuln_columns = [col for col in df.columns if any(word in col.lower() 
                for word in ['vuln', 'bug', 'issue', 'security', 'exploit', 'attack'])]

if len(vuln_columns) == 0:
    print("⚠️  NO VULNERABILITY LABELS FOUND IN THIS DATASET!")
    print("\nThis dataset contains:")
    print("  ✅ Smart contract source code (47,398 contracts)")
    print("  ✅ Metadata (transactions, compiler, balance)")
    print("  ❌ NO vulnerability labels")
    print("\n🔍 This is a raw, unlabeled dataset from the wild!")
else:
    print(f"✅ Found {len(vuln_columns)} vulnerability-related columns:")
    for col in vuln_columns:
        print(f"  - {col}")

## 3. Check for SmartBugs Analysis Results

The README mentions SmartBugs was used to analyze this dataset. Let's check if those results are available.

In [ ]:
# Check for any analysis results files
result_files = list(DATA_DIR.glob('*results*')) + list(DATA_DIR.glob('*analysis*'))
print(f"Result files found: {len(result_files)}")
for f in result_files:
    print(f"  - {f.name}")

# Check for JSON files that might contain analysis
json_files = list(DATA_DIR.glob('*.json'))
print(f"\nJSON files found: {len(json_files)}")
for f in json_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.2f} MB)")

In [ ]:
# Check duplicates.json - this might have useful info
if (DATA_DIR / 'duplicates.json').exists():
    with open(DATA_DIR / 'duplicates.json', 'r') as f:
        duplicates = json.load(f)
    print(f"Duplicates file structure: {type(duplicates)}")
    if isinstance(duplicates, dict):
        print(f"Number of entries: {len(duplicates)}")
        print(f"Sample keys: {list(duplicates.keys())[:5]}")

## 4. Data Quality Checks

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())
print(f"\nPercentage of missing values:")
print(df.isnull().sum() / len(df) * 100)

In [ ]:
# Check compiler versions
print("Compiler Version Distribution:")
compiler_counts = df['compiler_version'].value_counts()
print(f"\nTotal unique compiler versions: {len(compiler_counts)}")
print(f"\nTop 10 compiler versions:")
print(compiler_counts.head(10))

# Plot compiler version distribution
plt.figure(figsize=(12, 6))
compiler_counts.head(20).plot(kind='bar')
plt.title('Top 20 Compiler Versions')
plt.xlabel('Compiler Version')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Check transaction counts
print("Transaction Statistics:")
print(df['nb_transaction'].describe())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(df['nb_transaction'], bins=50, log=True)
plt.xlabel('Number of Transactions')
plt.ylabel('Count (log scale)')
plt.title('Distribution of Transaction Counts')

plt.subplot(1, 2, 2)
plt.boxplot(df['nb_transaction'])
plt.ylabel('Number of Transactions')
plt.title('Transaction Count Boxplot')
plt.tight_layout()
plt.show()

In [ ]:
# Sample and read a few contracts to check quality
sample_contracts = np.random.choice(contract_files, min(5, len(contract_files)), replace=False)

print("Sample Contract Analysis:")
contract_stats = []

for contract_path in sample_contracts:
    try:
        with open(contract_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        
        lines = content.split('\n')
        non_empty_lines = [l for l in lines if l.strip()]
        
        stats = {
            'file': contract_path.name,
            'size_kb': contract_path.stat().st_size / 1024,
            'total_lines': len(lines),
            'non_empty_lines': len(non_empty_lines),
            'has_pragma': 'pragma' in content.lower(),
            'has_contract': 'contract' in content.lower(),
            'has_function': 'function' in content.lower()
        }
        contract_stats.append(stats)
        
    except Exception as e:
        print(f"Error reading {contract_path.name}: {e}")

stats_df = pd.DataFrame(contract_stats)
print(stats_df)

In [ ]:
# Check nb_lines.csv if available
if (DATA_DIR / 'nb_lines.csv').exists():
    lines_df = pd.read_csv(DATA_DIR / 'nb_lines.csv')
    print("Lines of Code Statistics:")
    print(lines_df.describe())
    
    if 'nb_lines' in lines_df.columns:
        plt.figure(figsize=(10, 5))
        plt.hist(lines_df['nb_lines'], bins=100, log=True)
        plt.xlabel('Lines of Code')
        plt.ylabel('Count (log scale)')
        plt.title('Distribution of Contract Size (LOC)')
        plt.show()

## 5. Summary and Recommendations

### Key Findings:

In [ ]:
print("="*80)
print("DATASET VALIDATION SUMMARY")
print("="*80)

print("\n📊 DATASET OVERVIEW:")
print(f"  ✅ Total contracts: {len(df):,}")
print(f"  ✅ Actual .sol files: {len(contract_files):,}")
print(f"  ✅ Metadata available: {df.shape[1]} columns")
print(f"  ✅ Date range: {df['creation_date'].min()} to {df['creation_date'].max()}")

print("\n🚨 CRITICAL ISSUE - NO LABELS:")
print("  ❌ This dataset has NO vulnerability labels")
print("  ❌ Cannot be used directly for supervised learning")
print("  ⚠️  You need to either:")
print("     1. Get SmartBugs analysis results (from smartbugs/smartbugs-results repo)")
print("     2. Use a different labeled dataset (like SmartBugs Curated)")
print("     3. Run analysis tools yourself to generate labels")

print("\n💡 WHAT THIS DATASET IS GOOD FOR:")
print("  ✅ Unsupervised learning (pre-training, clustering)")
print("  ✅ Large-scale pre-training for CodeBERT")
print("  ✅ Understanding Solidity code patterns")
print("  ✅ Feature extraction practice")
print("  ❌ NOT for supervised vulnerability detection (no labels!)")

print("\n🎯 RECOMMENDED NEXT STEPS:")
print("  1. Clone SmartBugs analysis results repository")
print("  2. Download SmartBugs Curated dataset (subset with labels)")
print("  3. Use this dataset for pre-training, labeled dataset for fine-tuning")
print("  4. Consider using SolidiFI dataset (has vulnerability labels)")
print("\n" + "="*80)

## 6. Check for Alternative Label Sources

In [ ]:
print("ALTERNATIVE DATASETS WITH LABELS:\n")
print("1. SmartBugs Curated:")
print("   URL: https://github.com/smartbugs/smartbugs-curated")
print("   Contains: ~143 vulnerable contracts with ground truth labels")
print("   Categories: Reentrancy, Integer Overflow, etc.")
print()
print("2. SolidiFI Dataset:")
print("   URL: https://github.com/DependableSystemsLab/SolidiFI")
print("   Contains: 9,369 buggy contracts with 7,661 bugs")
print("   Features: Injected bugs with labels")
print()
print("3. SmartBugs Results:")
print("   URL: https://github.com/smartbugs/smartbugs-results")
print("   Contains: Analysis results for SmartBugs Wild dataset")
print("   Features: Multiple tool outputs (Slither, Mythril, etc.)")
print()
print("4. Ethereum Smart Contract Dataset (Kaggle):")
print("   URL: Search for 'smart contract vulnerabilities' on Kaggle")
print("   Note: May have labeled datasets")